# FHIR Condition – Incremental Ingestion

Fetch paginated Condition resources from the public FHIR R4 API, preserve each original JSON Bundle in the Lakehouse Raw layer, and merge resource versions into the Bronze Delta table.

This notebook uses a persistent watermark, a one-hour overlap window, run-specific raw folders, and ingestion metadata logging. The watermark advances only after the Bronze merge succeeds.

## Cell 1 – Create shared control tables

In [1]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.fhir_watermark (
    resource_type STRING,
    last_successful_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.fhir_ingestion_log (
    run_id STRING,
    resource_name STRING,
    api_url_or_params STRING,
    extraction_timestamp TIMESTAMP,
    load_timestamp TIMESTAMP,
    pages_fetched BIGINT,
    records_fetched BIGINT,
    records_saved BIGINT,
    raw_folder_path STRING,
    execution_status STRING,
    error_message STRING
)
USING DELTA
""")

print("Watermark and ingestion log tables are ready.")

StatementMeta(, 5fd8c507-30bc-4d12-8909-d5a71b8bc4b9, 3, Finished, Available, Finished, False)

Watermark and ingestion log tables are ready.


## Cell 2 – Imports, configuration and watermark

In [2]:
import requests
import json
import uuid
from datetime import datetime, timezone, timedelta

from pyspark.sql.functions import (
    col, explode, to_json, current_timestamp, lit, input_file_name
)
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType
)
from delta.tables import DeltaTable

# FHIR API CONFIGURATION
FHIR_BASE_URL = "https://hapi.fhir.org/baseR4"
RESOURCE_TYPE = "Condition"
PAGE_SIZE = 100
OVERLAP_HOURS = 1

# CREATE A UNIQUE ID FOR THIS RUN
run_id = str(uuid.uuid4())
extraction_timestamp = datetime.now(timezone.utc)
extraction_timestamp_naive = extraction_timestamp.replace(tzinfo=None)

# READ THE LAST SUCCESSFUL WATERMARK
watermark_row = spark.sql(f"""
    SELECT last_successful_timestamp
    FROM bronze.fhir_watermark
    WHERE resource_type = '{RESOURCE_TYPE}'
""").first()

# FIRST RUN: LAST THREE DAYS
# LATER RUNS: ONE-HOUR OVERLAP FROM LAST SUCCESSFUL WATERMARK
if watermark_row is None or watermark_row["last_successful_timestamp"] is None:
    start_date = extraction_timestamp - timedelta(days=3)
else:
    last_successful = watermark_row["last_successful_timestamp"].replace(
        tzinfo=timezone.utc
    )
    start_date = last_successful - timedelta(hours=OVERLAP_HOURS)

start_time = start_date.strftime("%Y-%m-%dT%H:%M:%SZ")
end_time = extraction_timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")

api_url = f"{FHIR_BASE_URL}/{RESOURCE_TYPE}"
params = [
    ("_count", str(PAGE_SIZE)),
    ("_lastUpdated", f"ge{start_time}"),
    ("_lastUpdated", f"lt{end_time}"),
    ("_format", "json")
]

headers = {"Accept": "application/fhir+json"}
api_details = f"{api_url} | Parameters: {json.dumps(params)}"

# UNIQUE RUN FOLDER PREVENTS SAME-DAY RUNS FROM OVERWRITING RAW FILES
extraction_date = extraction_timestamp.strftime("%Y-%m-%d")
raw_folder = (
    f"Files/raw/{RESOURCE_TYPE}/"
    f"extraction_date={extraction_date}/"
    f"run_id={run_id}"
)

print("Resource:", RESOURCE_TYPE)
print("Run ID:", run_id)
print("Start date:", start_time)
print("End date:", end_time)
print("Raw folder:", raw_folder)

StatementMeta(, 5fd8c507-30bc-4d12-8909-d5a71b8bc4b9, 4, Finished, Available, Finished, False)

Resource: Condition
Run ID: ed39ebf2-ffa8-4d56-a91b-6ac21ba643fa
Start date: 2026-11-18T21:05:20Z
End date: 2026-11-18T22:44:30Z
Raw folder: Files/raw/Condition/extraction_date=2026-11-18/run_id=ed39ebf2-ffa8-4d56-a91b-6ac21ba643fa


## Cell 3 – Metadata logging helper and start the run

In [3]:
log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("resource_name", StringType(), False),
    StructField("api_url_or_params", StringType(), True),
    StructField("extraction_timestamp", TimestampType(), True),
    StructField("load_timestamp", TimestampType(), True),
    StructField("pages_fetched", LongType(), True),
    StructField("records_fetched", LongType(), True),
    StructField("records_saved", LongType(), True),
    StructField("raw_folder_path", StringType(), True),
    StructField("execution_status", StringType(), False),
    StructField("error_message", StringType(), True)
])

def write_ingestion_log(status, pages=0, fetched=0, saved=0, error_message=None):
    log_row = [(
        run_id,
        RESOURCE_TYPE,
        api_details,
        extraction_timestamp_naive,
        datetime.now(timezone.utc).replace(tzinfo=None),
        int(pages),
        int(fetched),
        int(saved),
        raw_folder,
        status,
        error_message
    )]

    log_df = spark.createDataFrame(log_row, schema=log_schema)
    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("bronze.fhir_ingestion_log")
    )

write_ingestion_log("STARTED")
print("Ingestion run logged.")

StatementMeta(, 5fd8c507-30bc-4d12-8909-d5a71b8bc4b9, 5, Finished, Available, Finished, False)

Ingestion run logged.


## Cell 4 – Fetch paginated API responses and save original JSON

In [4]:
notebookutils.fs.mkdirs(raw_folder)

next_url = api_url
next_params = params
page_number = 1
total_records = 0

try:
    while next_url:
        print(f"Fetching page {page_number}...")

        response = requests.get(
            next_url,
            headers=headers,
            params=next_params,
            timeout=120
        )
        response.raise_for_status()

        # SAVE THE ORIGINAL RESPONSE TEXT WITHOUT MODIFYING IT
        raw_json = response.text
        bundle = response.json()

        file_path = f"{raw_folder}/page_{page_number:04d}.json"
        notebookutils.fs.put(file_path, raw_json, True)

        records_in_page = len(bundle.get("entry", []))
        total_records += records_in_page
        print(f"Page {page_number}: {records_in_page} records saved")

        next_link = next(
            (
                link["url"]
                for link in bundle.get("link", [])
                if link.get("relation") == "next"
            ),
            None
        )

        # NEXT LINK ALREADY CONTAINS ITS QUERY PARAMETERS
        next_url = next_link
        next_params = None
        page_number += 1

    total_pages = page_number - 1
    print("Pages fetched:", total_pages)
    print("Records fetched:", total_records)
    print("Raw folder:", raw_folder)

except Exception as exc:
    write_ingestion_log(
        "FAILED",
        pages=page_number - 1,
        fetched=total_records,
        error_message=str(exc)[:4000]
    )
    raise

StatementMeta(, 5fd8c507-30bc-4d12-8909-d5a71b8bc4b9, 6, Finished, Available, Finished, False)

Fetching page 1...
Page 1: 0 records saved
Pages fetched: 1
Records fetched: 0
Raw folder: Files/raw/Condition/extraction_date=2026-11-18/run_id=ed39ebf2-ffa8-4d56-a91b-6ac21ba643fa


## Cell 5 – Read the raw JSON Bundles

In [5]:
raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(raw_folder)
)

print("Raw Bundle files loaded.")
raw_df.printSchema()

StatementMeta(, 5fd8c507-30bc-4d12-8909-d5a71b8bc4b9, 7, Finished, Available, Finished, False)

Raw Bundle files loaded.
root
 |-- id: string (nullable = true)
 |-- link: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- relation: string (nullable = true)
 |    |    |-- url: string (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- lastUpdated: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- total: long (nullable = true)
 |-- type: string (nullable = true)



## Cell 6 – Extract individual Condition resources

In [6]:

# Extract individual Condition resources safely

if "entry" in raw_df.columns:

    condition_df = (
        raw_df
        .select(explode(col("entry")).alias("entry"))
        .select(col("entry.resource").alias("resource"))
    )

    print("Condition resources extracted.")
    print("Resource records:", condition_df.count())

else:
    print("No entry column found in the FHIR response.")
    print("The API may have returned an empty Bundle.")
    condition_df = None


StatementMeta(, 5fd8c507-30bc-4d12-8909-d5a71b8bc4b9, 8, Finished, Available, Finished, False)

No entry column found in the FHIR response.
The API may have returned an empty Bundle.


## Cell 7 – Prepare the consistent seven-column Bronze DataFrame

In [7]:

# KEEP THE COMPLETE FHIR RESOURCE IN resource_json

if condition_df is not None:

    bronze_condition_df = (
        condition_df
        .select(
            col("resource.*"),
            to_json(col("resource")).alias("resource_json")
        )
        .withColumn(
            "extraction_timestamp",
            lit(extraction_timestamp_naive).cast("timestamp")
        )
        .withColumn("api_url_or_params", lit(api_details))
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", input_file_name())
    )

    bronze_condition_final = (
        bronze_condition_df
        .select(
            "id",
            "resourceType",
            "resource_json",
            "extraction_timestamp",
            "api_url_or_params",
            "load_timestamp",
            "source_file"
        )
        .dropDuplicates(["id", "resource_json"])
    )

    print("Bronze DataFrame prepared.")
    print("Incoming records:", bronze_condition_final.count())
    bronze_condition_final.printSchema()

else:
    bronze_condition_final = None
    print("No Condition records received.")
    print("Bronze DataFrame preparation skipped.")


StatementMeta(, 5fd8c507-30bc-4d12-8909-d5a71b8bc4b9, 9, Finished, Available, Finished, False)

No Condition records received.
Bronze DataFrame preparation skipped.


## Cell 8 – Merge Bronze data, then advance the watermark

In [8]:

try:
    target_table = "bronze.condition"
    records_saved = 0

    # Merge only when new Condition records are available
    if bronze_condition_final is not None and bronze_condition_final.limit(1).count() > 0:

        if not spark.catalog.tableExists(target_table):
            (
                bronze_condition_final.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(target_table)
            )
            records_saved = bronze_condition_final.count()

        else:
            bronze_table = DeltaTable.forName(spark, target_table)

            before_count = spark.table(target_table).count()

            (
                bronze_table.alias("target")
                .merge(
                    bronze_condition_final.alias("source"),
                    """
                    target.id = source.id
                    AND target.resource_json = source.resource_json
                    """
                )
                .whenNotMatchedInsert(
                    values={
                        "id": "source.id",
                        "resourceType": "source.resourceType",
                        "resource_json": "source.resource_json",
                        "extraction_timestamp": "source.extraction_timestamp",
                        "api_url_or_params": "source.api_url_or_params",
                        "load_timestamp": "source.load_timestamp",
                        "source_file": "source.source_file"
                    }
                )
                .execute()
            )

            after_count = spark.table(target_table).count()
            records_saved = after_count - before_count

        print("Bronze Condition merge completed.")

    else:
        print("No new Condition records available.")
        print("Existing Bronze data will remain unchanged.")

    # Update watermark after successful API and raw-data processing
    watermark_update_df = spark.createDataFrame(
        [(RESOURCE_TYPE, extraction_timestamp_naive)],
        ["resource_type", "last_successful_timestamp"]
    )

    watermark_table = DeltaTable.forName(spark, "bronze.fhir_watermark")

    (
        watermark_table.alias("target")
        .merge(
            watermark_update_df.alias("source"),
            "target.resource_type = source.resource_type"
        )
        .whenMatchedUpdate(
            set={
                "last_successful_timestamp": "source.last_successful_timestamp",
                "updated_at": "current_timestamp()"
            }
        )
        .whenNotMatchedInsert(
            values={
                "resource_type": "source.resource_type",
                "last_successful_timestamp": "source.last_successful_timestamp",
                "updated_at": "current_timestamp()"
            }
        )
        .execute()
    )

    write_ingestion_log(
        "SUCCESS",
        pages=total_pages,
        fetched=total_records,
        saved=records_saved
    )

    print("Condition watermark updated successfully.")
    print("New resource versions inserted:", records_saved)

except Exception as exc:

    write_ingestion_log(
        "FAILED",
        pages=locals().get("total_pages", 0),
        fetched=locals().get("total_records", 0),
        saved=locals().get("records_saved", 0),
        error_message=str(exc)[:4000]
    )

    raise


StatementMeta(, 5fd8c507-30bc-4d12-8909-d5a71b8bc4b9, 10, Finished, Available, Finished, False)

No new Condition records available.
Existing Bronze data will remain unchanged.
Condition watermark updated successfully.
New resource versions inserted: 0


## Cell 9 – Verify Bronze, watermark and ingestion log

In [9]:
print("BRONZE CONDITION VALIDATION")

spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT id) AS unique_ids,
        COUNT(*) - COUNT(id) AS null_ids
    FROM bronze.condition
""").show()

print("WATERMARK")
spark.sql("""
    SELECT *
    FROM bronze.fhir_watermark
    WHERE resource_type = 'Condition'
""").show(truncate=False)

print("INGESTION LOG")
spark.sql(f"""
    SELECT *
    FROM bronze.fhir_ingestion_log
    WHERE run_id = '{run_id}'
    ORDER BY load_timestamp DESC
""").show(truncate=False)

StatementMeta(, 5fd8c507-30bc-4d12-8909-d5a71b8bc4b9, 11, Finished, Available, Finished, False)

BRONZE CONDITION VALIDATION
+-------------+----------+--------+
|total_records|unique_ids|null_ids|
+-------------+----------+--------+
|        11878|     11878|       0|
+-------------+----------+--------+

WATERMARK
+-------------+--------------------------+--------------------------+
|resource_type|last_successful_timestamp |updated_at                |
+-------------+--------------------------+--------------------------+
|Condition    |2026-11-18 22:44:30.759343|2026-11-18 22:48:03.305481|
+-------------+--------------------------+--------------------------+

INGESTION LOG
+------------------------------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+--------------------------+-------------+---------------+-------------+------------------------------------------------------------------------------------